# Construir com os Modelos da Família Meta 

## Introdução 

Esta lição irá cobrir: 

- Explorar os dois principais modelos da família Meta - Llama 3.1 e Llama 3.2 
- Compreender os casos de uso e cenários para cada modelo 
- Exemplo de código para mostrar as características únicas de cada modelo 


## A Família de Modelos Meta 

Nesta lição, vamos explorar 2 modelos da família Meta ou "Rebanho Llama" - Llama 3.1 e Llama 3.2 

Estes modelos estão disponíveis em diferentes variantes e podem ser encontrados no [catálogo Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Nota:** O GitHub Models será descontinuado no final de julho de 2026. Aqui estão mais detalhes sobre como usar os [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) para prototipar com modelos de IA.

Variantes de Modelo: 
- Llama 3.1 - 70B Instruct 
- Llama 3.1 - 405B Instruct 
- Llama 3.2 - 11B Vision Instruct 
- Llama 3.2 - 90B Vision Instruct 

*Nota: Llama 3 também está disponível no Microsoft Foundry Models, mas não será abordado nesta lição*



## Llama 3.1 

Com 405 mil milhões de parâmetros, o Llama 3.1 enquadra-se na categoria de LLM de código aberto. 

O modelo é uma atualização da versão anterior Llama 3, oferecendo: 

- Janela de contexto maior - 128k tokens vs 8k tokens 
- Máximo de Tokens de Saída maior - 4096 vs 2048 
- Melhor suporte multilíngue - devido ao aumento dos tokens de treino 

Estes permitem que o Llama 3.1 lide com casos de uso mais complexos na construção de aplicações GenAI, incluindo: 
- Chamada de Funções Nativas - a capacidade de chamar ferramentas e funções externas fora do fluxo de trabalho do LLM
- Melhor desempenho em RAG - devido à maior janela de contexto 
- Geração de Dados Sintéticos - a capacidade de criar dados eficazes para tarefas como o fine-tuning 



### Chamada Nativa de Funções 

O Llama 3.1 foi ajustado para ser mais eficaz a fazer chamadas de funções ou ferramentas. Também tem duas ferramentas incorporadas que o modelo pode identificar como necessárias com base no prompt do utilizador. Estas ferramentas são: 

- **Brave Search** - Pode ser usado para obter informação atualizada, como o tempo, realizando uma pesquisa na web 
- **Wolfram Alpha** - Pode ser usado para cálculos matemáticos mais complexos, pelo que não é necessário escrever as suas próprias funções. 

Também pode criar as suas próprias ferramentas personalizadas que o LLM pode chamar. 

No exemplo de código abaixo: 

- Definimos as ferramentas disponíveis (brave_search, wolfram_alpha) no prompt do sistema. 
- Enviamos um prompt do utilizador que pergunta sobre o tempo numa determinada cidade. 
- O LLM irá responder com uma chamada à ferramenta Brave Search que terá este aspeto `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Nota: Este exemplo apenas faz a chamada da ferramenta, se quiser obter os resultados, terá de criar uma conta gratuita na página da API da Brave e definir a função propriamente dita* 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Apesar de ser um LLM, uma limitação que o Llama 3.1 tem é a multimodalidade. Ou seja, conseguir utilizar diferentes tipos de entrada, como imagens, como prompts e fornecer respostas. Esta capacidade é uma das principais funcionalidades do Llama 3.2. Estas funcionalidades incluem também: 

- Multimodalidade - tem a capacidade de avaliar prompts tanto de texto como de imagem 
- Variações de tamanho pequeno a médio (11B e 90B) - isto fornece opções flexíveis de implementação, 
- Variações só de texto (1B e 3B) - isto permite que o modelo seja implementado em dispositivos edge / móveis e oferece baixa latência 

O suporte multimodal representa um grande passo no mundo dos modelos open source. O exemplo de código abaixo utiliza tanto uma imagem como um prompt de texto para obter uma análise da imagem pelo Llama 3.2 90B. 

### Suporte Multimodal com Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## A aprendizagem não termina aqui, continue a Jornada

Depois de completar esta lição, consulte a nossa [coleção de Aprendizagem em IA Generativa](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) para continuar a aumentar o seu conhecimento em IA Generativa!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
